# 04 - Train Neural Network (MLP)

Train a Multi-Layer Perceptron (feedforward neural network) for toxicity prediction.

**Architecture Options:**
- Hidden layers: (64,), (128,), (64,32), (128,64), etc.
- Activation: ReLU, Tanh
- Early stopping and dropout regularization

In [1]:
# ============================================================================
# IMPORTS AND CONFIG
# ============================================================================
import os, sys, pickle
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Hyperparameter grid
PARAM_GRID = {
    'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [1000],
    'early_stopping': [True]
}

TOXICITY_ENDPOINTS = ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-Aromatase',
                      'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
                      'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
MODELS_DIR = '../models/baseline_models'
print("✓ Setup complete")

✓ Setup complete


In [ ]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_nn(toxicity_name):
    """Train Neural Network for a single toxicity endpoint."""
    print(f"\nTraining NN for {toxicity_name}...")
    
    cache_path = f'../Data/cache/{toxicity_name}/splits.pkl'
    if not os.path.exists(cache_path):
        print(f"⚠️ Data not found")
        return None
    
    with open(cache_path, 'rb') as f:
        data = pickle.load(f)
    
    X_train_val = np.vstack([data['train']['X'], data['val']['X']])
    y_train_val = np.concatenate([data['train']['y'], data['val']['y']])
    X_test, y_test = data['test']['X'], data['test']['y']
    
    # Grid search
    nn = MLPClassifier(random_state=42)
    grid_search = GridSearchCV(nn, PARAM_GRID, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train_val, y_train_val)
    
    # Evaluate
    best_model = grid_search.best_estimator_
    y_proba = best_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    
    print(f"  Best: {grid_search.best_params_['hidden_layer_sizes']}, AUC: {test_auc:.4f}")
    
    # Save
    os.makedirs(f'{MODELS_DIR}/{toxicity_name}', exist_ok=True)
    with open(f'{MODELS_DIR}/{toxicity_name}/NN_model.pkl', 'wb') as f:
        pickle.dump({'model': best_model}, f)
    
    return {'test_auc': test_auc}

# Example
result = train_nn('NR-AhR')


Training NN for NR-AhR...
